# CarePath DARAG - Part 1: Data Prep (CPU)

Runs on **Colab (CPU runtime) or any local machine** - the setup cell auto-detects.
The slow step is real Gipformer ASR over ViMedCSS to build `raw_asr -> gold_text`
pairs (paper Sec 3.1); it is CPU-bound, so on Colab use a **CPU runtime** to avoid
spending GPU units. Outputs are saved to Google Drive on Colab, or kept in the
repo's `artifacts/` locally - Part 2 picks them up either way.

In [ ]:
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer

## 1. Set up the environment (Colab or local)

In [ ]:
# Set up the repo path + detect the runtime. Works on Colab AND a local machine.
import importlib.util, os, shutil, subprocess, sys, urllib.parse, zipfile
from pathlib import Path

DEFAULT_CAREPATH_REPO_URL = 'https://github.com/truong-tt/carepath.git'

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

def is_carepath_repo(path):
    path = Path(path)
    return (path / 'pyproject.toml').exists() and (path / 'apps' / 'api' / 'carepath').exists()

def remove_path(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()

def find_carepath_repo(start):
    start = Path(start).resolve()
    for d in [start, *start.parents]:
        if is_carepath_repo(d):
            return d
    if IN_COLAB:
        candidate_values = [
            os.environ.get('CAREPATH_REPO_DIR'),
            '/content/carepath',
            '/content/CarePath',
            '/content/drive/MyDrive/carepath',
            '/content/drive/MyDrive/CarePath',
        ]
        for value in candidate_values:
            if value and is_carepath_repo(Path(value)):
                return Path(value)
    return None

def extract_repo_zip(zip_path, target):
    zip_path = Path(zip_path).expanduser()
    target = Path(target).expanduser()
    extract_root = Path('/content/carepath_unzip') if IN_COLAB else target.parent / 'carepath_unzip'
    remove_path(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_root)
    roots = sorted({p.parent for p in extract_root.rglob('pyproject.toml')}, key=lambda p: len(p.parts))
    for src in roots:
        if is_carepath_repo(src):
            remove_path(target)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src, target)
            return target
    raise SystemExit(f'{zip_path} did not contain a CarePath repo.')

def upload_repo_zip():
    try:
        from google.colab import files
    except Exception as exc:
        raise SystemExit('Colab: upload carepath.zip to /content, or set CAREPATH_REPO_ZIP / CAREPATH_REPO_URL.') from exc
    print('Upload carepath.zip when prompted, or rerun after setting CAREPATH_REPO_ZIP / CAREPATH_REPO_URL.')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file was uploaded. Upload carepath.zip to /content, or set CAREPATH_REPO_ZIP / CAREPATH_REPO_URL.')
    for name in uploaded:
        path = Path(name)
        if not path.is_absolute():
            path = Path.cwd() / path
        if path.suffix.lower() == '.zip':
            return path
    raise SystemExit('Uploaded file must be a .zip containing the CarePath repo.')

def get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None

def github_clone_url(repo_url, token):
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    parsed = urllib.parse.urlsplit(repo_url)
    auth = 'x-access-token:' + urllib.parse.quote(token, safe='')
    return urllib.parse.urlunsplit((parsed.scheme, f'{auth}@{parsed.netloc}', parsed.path, parsed.query, parsed.fragment))

def redact_git_output(text, token):
    if not text:
        return ''
    if token:
        text = text.replace(token, '***').replace(urllib.parse.quote(token, safe=''), '***')
    return text.strip()

REPO = find_carepath_repo(Path.cwd())
if REPO is None and IN_COLAB:
    target = Path(os.environ.get('CAREPATH_REPO_DIR') or '/content/carepath')
    repo_url = os.environ.get('CAREPATH_REPO_URL') or DEFAULT_CAREPATH_REPO_URL
    zip_candidates = []
    if os.environ.get('CAREPATH_REPO_ZIP'):
        zip_candidates.append(Path(os.environ['CAREPATH_REPO_ZIP']))
    zip_candidates.extend([
        Path('/content/carepath.zip'),
        Path('/content/CarePath.zip'),
        Path('/content/drive/MyDrive/carepath.zip'),
        Path('/content/drive/MyDrive/CarePath.zip'),
        Path('/content/drive/MyDrive/CarePath/carepath.zip'),
    ])
    content_dir = Path('/content')
    if content_dir.exists():
        zip_candidates.extend(sorted(content_dir.glob('*carepath*.zip')))
        zip_candidates.extend(sorted(content_dir.glob('*CarePath*.zip')))
    seen = set()
    for zip_path in zip_candidates:
        zip_path = Path(zip_path).expanduser()
        if zip_path in seen:
            continue
        seen.add(zip_path)
        if zip_path.exists():
            REPO = extract_repo_zip(zip_path, target)
            print('Extracted repo zip:', zip_path)
            break
    if REPO is None and repo_url:
        remove_path(target)
        github_token = get_secret('CAREPATH_GITHUB_TOKEN') or get_secret('GITHUB_TOKEN')
        clone_url = github_clone_url(repo_url, github_token)
        result = subprocess.run(['git', 'clone', clone_url, str(target)], text=True, capture_output=True)
        if result.returncode == 0:
            REPO = target
            suffix = ' using GitHub token' if github_token else ''
            print('Cloned repo:', repo_url + suffix)
        else:
            git_output = redact_git_output(result.stderr or result.stdout, github_token)
            raise SystemExit(
                'Could not clone CarePath from GitHub. This repo is likely private to the Colab runtime.\n'
                f'Repo URL: {repo_url}\n'
                f'Git output: {git_output}\n\n'
                'Fix: make the repo public, or add a Colab Secret named GITHUB_TOKEN or CAREPATH_GITHUB_TOKEN '
                'with read access to this repo, then rerun this cell. You can still set CAREPATH_REPO_ZIP '
                'to a Drive zip path if you prefer the zip route.'
            )
    if REPO is None and os.environ.get('CAREPATH_UPLOAD_FALLBACK') == '1':
        REPO = extract_repo_zip(upload_repo_zip(), target)

if REPO is None:
    raise SystemExit('Could not find the CarePath repo. Open this notebook from inside the cloned repo.')

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'apps' / 'api'))
print(('Colab' if IN_COLAB else 'Local'), '| repo:', REPO)


In [ ]:
# Helper: run a pipeline CLI with PYTHONPATH set, streaming output, raising on failure.
import os, subprocess, sys

def run_step(args, env_extra=None):
    env = dict(os.environ)
    env["PYTHONPATH"] = "apps/api"
    env["PYTHONIOENCODING"] = "utf-8"
    if env_extra:
        env.update(env_extra)
    print(">>>", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed ({proc.returncode}): {' '.join(args)}")

In [ ]:
from carepath.gec.env import setup_backup
# Part 1 is CPU-only (Gipformer ASR), so no GPU is needed here.
BACKUP = setup_backup(IN_COLAB)

## 2. Run size - keep smoke defaults, raise for a real run

In [ ]:
LIMIT_PER_SPLIT = 20   # None for the full dataset
DATASET = 'tensorxt/ViMedCSS'
DATASTORE = 'artifacts/retrieval/term_datastore.json'
PAIRS = 'artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl'
print('LIMIT_PER_SPLIT =', LIMIT_PER_SPLIT)

## 3. Build the NE / code-switch datastore (paper Sec 4.2 Step 1)

In [ ]:
run_step([
    "scripts/gec/build_datastore.py",
    "--dataset", DATASET,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--output", DATASTORE,
])

## 4. Build real Gipformer GEC pairs (CPU - the long step)

`--resume` makes this restartable if the runtime drops.

In [ ]:
run_step([
    "scripts/gec/make_pairs.py",
    "--dataset", DATASET,
    "--output", PAIRS,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--datastore", DATASTORE,
    "--resume",
])

## 5. Quick baseline WER on the raw ASR (paper Table 1 style)

In [ ]:
run_step([
    "scripts/gec/evaluate.py",
    "--input", PAIRS,
    "--prediction-columns", "raw_asr",
    "--wer-output", "artifacts/evaluations/raw_baseline_wer.json",
    "--ne-f1-output", "artifacts/evaluations/raw_baseline_ne_f1.json",
])

## 6. Save artifacts (Drive on Colab, disk locally)

In [ ]:
from carepath.gec.env import save_artifacts
save_artifacts(BACKUP, [DATASTORE, PAIRS, 'artifacts/evaluations/raw_baseline_wer.json'])
print('Part 1 done. On Colab the files are on Drive; locally they are in artifacts/.')
print('Next: open Part 2 on a GPU runtime (Colab L4 or a local NVIDIA GPU).')